In [1]:
# cnn_model.ipynb
# Day 4: Build 1D-CNN IDS model
# 1D-CNN = trees 80 features as sequence
# Learns local patterns in network traffic

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import(
    accuracy_score, precision_score,
    recall_score, f1_score,
    classification_report
)
import matplotlib.pyplot as plt
import time

print(f"TensorFlow version: {tf.__version__}")
print("Libraries loaded!")

2026-03-28 11:39:00.540543: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


TensorFlow version: 2.16.2
Libraries loaded!


In [5]:
# Step 2: Load preprocessed data
print("Loading data...")

save_path = "/Users/miuyanhong/Desktop/replication_studies/XAI_Feature_Selection/CICIDS-2017/processed/"

X_train = np.load(save_path + "X_train.npy")
X_test = np.load(save_path + "X_test.npy")
y_train = pd.read_csv(save_path + "y_train.csv").squeeze()
y_test =pd.read_csv(save_path + "y_test.csv").squeeze()

print(f"X_train: {X_train.shape}")
print(f"X_test: {X_test.shape}")
print("Data loaded!")

Loading data...
X_train: (1979513, 80)
X_test: (848363, 80)
Data loaded!


In [6]:
# Step 3: Encode labels
# CNN needs numbers not text labels
# BENIGN = 0, Bot = 1, DDOS 2 ect.
from sklearn.preprocessing import LabelEncoder

print("Encoding labels...")
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

num_classes = len(le.classes_)
print(f"Classes: {list(le.classes_)}")
print(f"Number of classes: {num_classes}")
print("Labels encoded!")


Encoding labels...
Classes: ['BENIGN', 'Bot', 'DDoS', 'DoS GoldenEye', 'DoS Hulk', 'DoS Slowhttptest', 'DoS slowloris', 'FTP-Patator', 'Heartbleed', 'Infiltration', 'PortScan', 'SSH-Patator', 'Web Attack']
Number of classes: 13
Labels encoded!


In [7]:
# Step 4: Reshape data for 1D_CNN
# CNN needs 3D input: (samples, steps, features)
# Current shape: (1979513,80)
# Need shape:    (1979513, 80, 1)

X_train_cnn = X_train.reshape(
    X_train.shape[0], X_train.shape[1],1)
X_test_cnn = X_test.reshape(
    X_test.shape[0], X_test.shape[1],1)

print(f"X_train, reshaped: {X_train_cnn.shape}")
print(f"X_test reshaped: {X_test_cnn.shape}")
print("Reshape complete!")


X_train, reshaped: (1979513, 80, 1)
X_test reshaped: (848363, 80, 1)
Reshape complete!


In [17]:
# Step 5: Build 1D-CNN model
# Conv1D = learn local feature patterns
# MaxPooling = reduces dimensions 
# Dense = final classification layers 

print("Building 1D-CNN model...")

model = keras.Sequential([
    # input layer
    keras.layers.Input(shape=(80,1)),

    # Covn layer 1: learn local patterns
    keras.layers.Conv1D(
        filters = 64,
        kernel_size = 3,
        activation = 'relu',
        padding = 'same'
    ),
    keras.layers.MaxPooling1D(pool_size=2),
    # Conv layer 2: learn deeper patterns 
    keras.layers.Conv1D(
        filters=123,
        kernel_size=3,
        activation='relu',
        padding='same'
    ),
    keras.layers.MaxPooling1D(pool_size=2),

    # Flatten and classify
    keras.layers.Flatten(),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(num_classes, activation='softmax')
])

# Compile model
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Building 1D-CNN model...


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_9 (Conv1D)               │ (None, 80, 64)         │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_9 (MaxPooling1D)  │ (None, 40, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_10 (Conv1D)              │ (None, 40, 123)        │        23,739 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_10 (MaxPooling1D) │ (None, 20, 123)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_4 (Flatten)             │ (None, 2460)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 128)            │       315,008 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 13)             │         1,677 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 340,680 (1.30 MB)

 Trainable params: 340,680 (1.30 MB)

 Non-trainable params: 0 (0.00 B)

In [22]:
# Step 6: Train 1D-CNN model

print("Training 1D-CNN...")
print("This will take 30-60 mitunes...")

start_time = time.time()

history = model.fit(
    X_train_cnn, y_train_enc,
    epochs=5,
    batch_size=1024,
    validation_split=0.1,
    verbose=1
)

end_time = time.time()
cnn_train_time = round(end_time - start_time, 2)

print(f"Training complete!")
print(f"Training time: {cnn_train_time} second")


Training 1D-CNN...
This will take 30-60 mitunes...
Epoch 1/5
1740/1740 ━━━━━━━━━━━━━━━━━━━━ 639s 366ms/step - accuracy: 0.9440 - loss: 0.1894 - val_accuracy: 0.9797 - val_loss: 0.0483
Epoch 2/5
1740/1740 ━━━━━━━━━━━━━━━━━━━━ 535s 308ms/step - accuracy: 0.9771 - loss: 0.0521 - val_accuracy: 0.9794 - val_loss: 0.0447
Epoch 3/5
1740/1740 ━━━━━━━━━━━━━━━━━━━━ 622s 357ms/step - accuracy: 0.9800 - loss: 0.0455 - val_accuracy: 0.9823 - val_loss: 0.0399
Epoch 4/5
1740/1740 ━━━━━━━━━━━━━━━━━━━━ 601s 345ms/step - accuracy: 0.9817 - loss: 0.0419 - val_accuracy: 0.9855 - val_loss: 0.0354
Epoch 5/5
1740/1740 ━━━━━━━━━━━━━━━━━━━━ 466s 268ms/step - accuracy: 0.9840 - loss: 0.0375 - val_accuracy: 0.9870 - val_loss: 0.0320
Training complete!
Training time: 2872.22 second


In [24]:
# Step 7: Evaluate CNN model
print("Evaluating CNN model...")

start_time = time.time()
y_pred_enc = model.predict(
    X_test_cnn, batch_size=1024, verbose=1)
y_pred = np.argmax(y_pred_enc, axis=1)
end_time = time.time()

cnn_pred_time = round(end_time - start_time,2)

# Convert back to labels
y_pred_labels = le.inverse_transform(y_pred)

# Calcucate metrics
acc = accuracy_score(y_test, y_pred_labels)
p   = precision_score(y_test, y_pred_labels,
                     average='weighted')
r   = recall_score(y_test, y_pred_labels,
                  average='weighted')
f1  = f1_score(y_test, y_pred_labels,
              average='weighted')
print(f"\n=== 1D-CNN Results ===")
print(f"Accuracy:  {acc:.4f} ({acc*100:.2f}%)")
print(f"Precision: {p:.4f} ({p*100:.2f}%)")
print(f"Recall:    {r:.4f} ({r*100:.2f}%)")
print(f"F1_score:  {f1:.3f} ({f1*100:.2f}%)")
print(f"Prediction time: {cnn_pred_time}s")
print(f"\nDetailed Report:")
print(classification_report(y_test, y_pred_labels))


Evaluating CNN model...
829/829 ━━━━━━━━━━━━━━━━━━━━ 68s 81ms/step


/opt/miniconda3/envs/cnn_env/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))



=== 1D-CNN Results ===
Accuracy:  0.9874 (98.74%)
Precision: 0.9879 (98.79%)
Recall:    0.9874 (98.74%)
F1_score:  0.987 (98.73%)
Prediction time: 69.89s

Detailed Report:


/opt/miniconda3/envs/cnn_env/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/miniconda3/envs/cnn_env/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


                  precision    recall  f1-score   support

          BENIGN       0.99      0.99      0.99    681396
             Bot       0.95      0.73      0.82       587
            DDoS       1.00      1.00      1.00     38408
   DoS GoldenEye       0.99      0.97      0.98      3088
        DoS Hulk       0.98      0.99      0.99     69037
DoS Slowhttptest       0.93      0.98      0.96      1650
   DoS slowloris       0.99      0.99      0.99      1739
     FTP-Patator       0.98      0.99      0.99      2380
      Heartbleed       0.00      0.00      0.00         3
    Infiltration       0.00      0.00      0.00        11
        PortScan       0.90      0.96      0.93     47641
     SSH-Patator       0.96      0.99      0.97      1769
      Web Attack       0.98      0.07      0.13       654

        accuracy                           0.99    848363
       macro avg       0.82      0.74      0.75    848363
    weighted avg       0.99      0.99      0.99    848363



/opt/miniconda3/envs/cnn_env/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [27]:
# Step 8: Save CNN results
import json

# Save model
model.save(save_path + "cnn_model.keras")

# Save model
cnn_results = {
    "model": "1D-CNN",
    "accuracy": acc,
    "precision": p,
    "recall": r,
    "f1_score": f1,
    "training_time": 2872.22,
    "prediction_time": 69.89,
    "epochs": 5,
    "batch_size": 1024
}

with open(save_path + "cnn_results.json", "w")as f:
    json.dump(cnn_results, f, indent=4)

print("CNN model saved!")

CNN model saved!
